# Libs

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import gc
import timeit
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, recall_score

# File Paths

In [19]:
MODELS_DIR = pathlib.Path("../Models")
RESULTS_DIR = pathlib.Path("../results")

DATA_PATH = pathlib.Path("../../../data/fdia_dataset_processed.npz")

RESULTS_DIR.mkdir(exist_ok=True)

print("Models:", MODELS_DIR.resolve())
print("Results:", RESULTS_DIR.resolve())
print("Dataset:", DATA_PATH.resolve())

Models: /home/lcdlu/tinyml-fdia/Models_Prunning/4-Inference_test/Models
Results: /home/lcdlu/tinyml-fdia/Models_Prunning/4-Inference_test/results
Dataset: /home/lcdlu/tinyml-fdia/data/fdia_dataset_processed.npz


# Dataset

In [18]:
data = np.load(DATA_PATH)

X_test = data["X_test"]
y_test = data["y_test"]

X_test_lstm = X_test.transpose(0, 2, 1).astype(np.float32)

print("X_test:", X_test.shape)
print("X_test_lstm:", X_test_lstm.shape)
print("y_test:", y_test.shape)

X_test: (9720, 6, 83)
X_test_lstm: (9720, 83, 6)
y_test: (9720,)


# Create subsamples

In [17]:
np.random.seed(42)

sample_size = int(0.05 * X_test_lstm.shape[0])

timing_idx = np.random.choice(
    X_test_lstm.shape[0],
    sample_size,
    replace=False
)

X_timing = X_test_lstm[timing_idx]

print("Total test samples:", len(X_test_lstm))
print("Timing samples:", len(X_timing))
print("Timing shape:", X_timing.shape)

Total test samples: 9720
Timing samples: 486
Timing shape: (486, 83, 6)


# Functions

In [16]:
def measure_inference_time(model, X_timing):
    times = []

    for sample in X_timing:
        sample = np.expand_dims(sample, axis=0)

        start = timeit.default_timer()

        model.predict(sample, verbose=0)

        end = timeit.default_timer()

        times.append((end - start) * 1000)

        gc.collect()

    return np.array(times)

def test_model(model_name, filename):
    path = MODELS_DIR / filename

    print("Testing:", model_name)

    model = tf.keras.models.load_model(path)

    # Full test set evaluation
    y_prob = model.predict(X_test_lstm, verbose=0)
    y_pred = (y_prob > 0.5).astype(int).flatten()

    accuracy = accuracy_score(y_test, y_pred)
    fdia_recall = recall_score(y_test, y_pred, pos_label=1)
    fault_recall = recall_score(y_test, y_pred, pos_label=0)

    # Inference timing
    times = measure_inference_time(model, X_timing)

    mean_time = np.mean(times)

    # Save individual inference times
    np.save(
        RESULTS_DIR / f"{model_name}_inference_times.npy",
        times
    )

    # Save model result
    result = pd.DataFrame([{
        "model": model_name,
        "accuracy": accuracy,
        "fdia_recall": fdia_recall,
        "fault_recall": fault_recall,
        "inference_time_ms": mean_time
    }])

    result.to_csv(
        RESULTS_DIR / f"{model_name}_keras.csv",
        index=False
    )

    print("Accuracy:", accuracy)
    print("FDIA recall:", fdia_recall)
    print("Fault recall:", fault_recall)
    print("Mean inference time:", mean_time, "ms")

    del model
    gc.collect()

    return result

# LSTm TESTS

## LSTM Original

In [14]:
MODEL_NAME = "LSTM_Original"
MODEL_FILE = "LSTM_HPO.keras"

result = test_model(
    MODEL_NAME,
    MODEL_FILE
)

print("\n=== FINAL RESULT ===")
print(result.to_string(index=False))

Testing: LSTM_Original


KeyboardInterrupt: 

# Diagnoses

In [22]:
MODEL_PATH = MODELS_DIR / "LSTM_HPO.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded:", MODEL_PATH)
print("Timing samples:", len(X_timing))

Model loaded: ../Models/LSTM_HPO.keras
Timing samples: 486


In [23]:
sample = np.expand_dims(X_timing[0], axis=0)

model.predict(sample, verbose=0)
model(sample, training=False).numpy()

gc.collect()

print("Warm-up done")

Warm-up done


In [ ]:
predict_times = []

for sample in X_timing:
    sample = np.expand_dims(sample, axis=0)

    start = timeit.default_timer()

    model.predict(sample, verbose=0)

    end = timeit.default_timer()

    predict_times.append((end - start) * 1000)

    gc.collect()

print("model.predict() mean:", np.mean(predict_times), "ms")

In [ ]:
direct_times = []

for sample in X_timing:
    sample = np.expand_dims(sample, axis=0)

    start = timeit.default_timer()

    model(sample, training=False).numpy()

    end = timeit.default_timer()

    direct_times.append((end - start) * 1000)

    gc.collect()

print("Direct call mean:", np.mean(direct_times), "ms")

In [ ]:
predict_mean = np.mean(predict_times)
direct_mean = np.mean(direct_times)

print("=== FINAL DIAGNOSTIC ===")
print(f"model.predict(): {predict_mean:.3f} ms")
print(f"Direct call:     {direct_mean:.3f} ms")
print(f"Ratio:           {predict_mean / direct_mean:.2f}x")